# Tema 2.2 - RetailMax
## Práctica guiada: selección, transformación, normalización y estandarización de variables

Este notebook mantiene la misma estructura homologada de los cuadernos anteriores y está organizado por **pasos**.

### Propósito de la práctica
En esta práctica trabajarás con el archivo `retailmax_integrado.csv`, que concentra múltiples fuentes de datos en un único **DataFrame**. A partir de este conjunto realizarás tres tareas fundamentales de preparación de datos:

- **Selección de variables**
- **Transformación de variables categóricas**
- **Normalización y estandarización**

### Estructura esperada del dataset
Cada fila representa la venta de un producto específico en una sucursal y una fecha determinada, incorporando variables como:

- `date`, `day_of_week`, `store_id`
- `product_id`, `category`
- `units`, `price`, `revenue`, `promo`, `inventory`
- `avg_temp`
- `topic1` a `topic5`
- `img_mean`, `img_std`

### Archivos de salida que generará este notebook
- `correlation_matrix.csv`
- `vif_results.csv`
- `selected_variables.csv`
- `retailmax_selected.csv`
- `retailmax_encoded.csv`
- `retailmax_minmax.csv`
- `retailmax_standardized.csv`


In [ ]:
# ============================================================
# 0. Configuración inicial
# ============================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

SEED = 42
rng = np.random.default_rng(SEED)

display(Markdown("### Librerías cargadas correctamente"))
print("Entorno listo para la práctica 5 de RetailMax.")


## Paso 1. Carga del dataset y exploración inicial

El primer paso en cualquier proceso de preparación de datos consiste en cargar el conjunto de datos y revisar su estructura general.

En este notebook se intentará cargar, en este orden:
1. `retailmax_integrado.csv`
2. `retailmax_master.csv`

Si ninguno se encuentra disponible en el entorno, se generará una **simulación de respaldo** con la misma estructura conceptual de RetailMax para que puedas continuar con la práctica.


In [ ]:
# ============================================================
# 1. Carga del dataset o simulación de respaldo
# ============================================================

candidate_files = ["retailmax_integrado.csv", "retailmax_master.csv"]
dataset_path = None

for file_name in candidate_files:
    if Path(file_name).exists():
        dataset_path = file_name
        break

if dataset_path is not None:
    retail_df = pd.read_csv(dataset_path)
    source_mode = f"archivo real: {dataset_path}"
else:
    # --------------------------------------------------------
    # Simulación de respaldo
    # --------------------------------------------------------
    dates = pd.date_range(end=pd.Timestamp.today().normalize(), periods=30, freq="D")
    stores = ["S1", "S2", "S3"]
    products = [
        ("P101", "Abarrotes"),
        ("P102", "Bebidas"),
        ("P103", "Botanas"),
        ("P104", "Cuidado personal"),
    ]

    rows = []
    for date in dates:
        day_of_week = date.day_name()
        avg_temp = rng.normal(30, 2)
        for store in stores:
            for product_id, category in products:
                base_units = rng.integers(20, 90)
                promo = rng.integers(0, 2)
                inventory = rng.integers(80, 300)

                if category == "Bebidas":
                    price = rng.normal(42, 3)
                elif category == "Abarrotes":
                    price = rng.normal(70, 4)
                elif category == "Botanas":
                    price = rng.normal(48, 3)
                else:
                    price = rng.normal(95, 5)

                topic1 = rng.normal(0.25, 0.08)
                topic2 = topic1 * 0.85 + rng.normal(0, 0.02)
                topic3 = rng.normal(0.10, 0.07)
                topic4 = rng.normal(-0.05, 0.06)
                topic5 = topic3 * 0.75 + rng.normal(0, 0.02)

                # Descriptores visuales casi constantes por producto para inducir baja utilidad
                img_mean_map = {"P101": 145, "P102": 165, "P103": 135, "P104": 155}
                img_std_map = {"P101": 18, "P102": 25, "P103": 20, "P104": 22}

                img_mean = img_mean_map[product_id] + rng.normal(0, 0.8)
                img_std = img_std_map[product_id] + rng.normal(0, 0.5)

                # Units con leve efecto de promo e inventario
                units = base_units + 6 * promo + 0.03 * inventory + rng.normal(0, 4)
                units = max(1, round(units))

                revenue = units * price

                rows.append({
                    "date": date,
                    "day_of_week": day_of_week,
                    "store_id": store,
                    "product_id": product_id,
                    "category": category,
                    "units": int(units),
                    "price": round(float(price), 2),
                    "revenue": round(float(revenue), 2),
                    "promo": int(promo),
                    "inventory": int(inventory),
                    "avg_temp": round(float(avg_temp), 2),
                    "topic1": round(float(topic1), 4),
                    "topic2": round(float(topic2), 4),
                    "topic3": round(float(topic3), 4),
                    "topic4": round(float(topic4), 4),
                    "topic5": round(float(topic5), 4),
                    "img_mean": round(float(img_mean), 4),
                    "img_std": round(float(img_std), 4),
                })

    retail_df = pd.DataFrame(rows)
    source_mode = "simulación de respaldo"

retail_df["date"] = pd.to_datetime(retail_df["date"], errors="coerce")

display(Markdown(f"### Fuente cargada: **{source_mode}**"))
display(Markdown("### Vista inicial del dataset"))
display(retail_df.head())

display(Markdown("### Dimensiones del dataset"))
print(retail_df.shape)

display(Markdown("### Tipos de dato"))
display(retail_df.dtypes.reset_index().rename(columns={"index": "variable", 0: "dtype"}))

display(Markdown("### Valores faltantes por columna"))
missing_df = retail_df.isna().sum().reset_index()
missing_df.columns = ["variable", "missing_values"]
display(missing_df)


## Paso 2. Identificación de variables numéricas y categóricas

Las técnicas de preparación de datos dependen del tipo de variable.

- El análisis de correlación se aplica sobre variables numéricas.
- Las variables categóricas requieren codificación.
- La normalización y la estandarización se aplican únicamente sobre variables escalares.

Además de la tipología estadística, en esta práctica se incorpora una lectura conceptual de relevancia para el negocio.


In [ ]:
# ============================================================
# 2. Identificación de variables numéricas y categóricas
# ============================================================

numeric_vars = retail_df.select_dtypes(include=[np.number]).columns.tolist()
categorical_vars = retail_df.select_dtypes(include=["object", "category"]).columns.tolist()

if "date" in retail_df.columns:
    # date se trata aparte como temporal
    pass

display(Markdown("### Variables numéricas"))
print(numeric_vars)

display(Markdown("### Variables categóricas"))
print(categorical_vars)

conceptual_importance_df = pd.DataFrame([
    {"variable": "units", "business_importance": "Alta", "comment": "Número de unidades vendidas; KPI central"},
    {"variable": "price", "business_importance": "Alta", "comment": "Influye directamente en la demanda y el revenue"},
    {"variable": "revenue", "business_importance": "Media", "comment": "Variable derivada de units × price, potencialmente redundante"},
    {"variable": "promo", "business_importance": "Alta", "comment": "Factor comercial clave"},
    {"variable": "inventory", "business_importance": "Media", "comment": "Influye en la disponibilidad del producto"},
    {"variable": "avg_temp", "business_importance": "Baja/variable", "comment": "En este escenario no necesariamente impacta las ventas"},
    {"variable": "topic1-topic5", "business_importance": "Media", "comment": "Representan componentes derivados de reseñas"},
    {"variable": "img_mean, img_std", "business_importance": "Muy baja", "comment": "No existe evidencia clara de impacto directo en ventas"}
])

display(Markdown("### Evaluación conceptual de variables"))
display(conceptual_importance_df)


## Paso 3. Selección de variables mediante correlación

La matriz de correlación permite detectar relaciones lineales entre variables numéricas y reconocer posibles redundancias.

En esta práctica se espera observar:
- relación fuerte entre `price` y `revenue`,
- correlaciones moderadas entre algunos tópicos textuales,
- y relaciones débiles o nulas en otras variables auxiliares.


In [ ]:
# ============================================================
# 3. Matriz de correlación
# ============================================================

corr_numeric_vars = [col for col in numeric_vars if col in retail_df.columns]
corr_df = retail_df[corr_numeric_vars].copy()

correlation_matrix = corr_df.corr(numeric_only=True)

display(Markdown("### Matriz de correlación"))
display(correlation_matrix.round(3))

# Visualización con matplotlib
plt.figure(figsize=(12, 8))
plt.imshow(correlation_matrix, aspect="auto")
plt.colorbar()
plt.xticks(range(len(correlation_matrix.columns)), correlation_matrix.columns, rotation=90)
plt.yticks(range(len(correlation_matrix.index)), correlation_matrix.index)
plt.title("Matriz de correlación - RetailMax")
plt.tight_layout()
plt.show()

# Pares con alta correlación
pairs = []
cols = correlation_matrix.columns.tolist()
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        corr_val = correlation_matrix.iloc[i, j]
        if abs(corr_val) >= 0.70:
            pairs.append({
                "variable_1": cols[i],
                "variable_2": cols[j],
                "correlation": round(float(corr_val), 4)
            })

high_corr_df = pd.DataFrame(pairs).sort_values(by="correlation", key=lambda s: s.abs(), ascending=False) if len(pairs) > 0 else pd.DataFrame(columns=["variable_1", "variable_2", "correlation"])

display(Markdown("### Pares con correlación alta (|r| >= 0.70)"))
display(high_corr_df if len(high_corr_df) > 0 else pd.DataFrame({"mensaje": ["No se detectaron pares altamente correlacionados con el umbral definido."]}))


## Paso 4. Selección basada en VIF

El **Variance Inflation Factor (VIF)** ayuda a detectar multicolinealidad entre variables predictoras numéricas.

### Criterio general de interpretación
- `VIF < 5`: aceptable
- `5 <= VIF < 10`: advertencia
- `VIF >= 10`: multicolinealidad alta

En esta práctica el VIF debe interpretarse en conjunto con el conocimiento del dominio, especialmente cuando hay variables derivadas de forma determinística, como `revenue = units × price`.


In [ ]:
# ============================================================
# 4. Cálculo de VIF
# ============================================================

def compute_vif(df_numeric: pd.DataFrame) -> pd.DataFrame:
    clean_df = df_numeric.copy()

    # Relleno simple para evitar errores
    for col in clean_df.columns:
        clean_df[col] = clean_df[col].fillna(clean_df[col].median())

    # Eliminar columnas constantes
    nunique = clean_df.nunique()
    valid_cols = nunique[nunique > 1].index.tolist()
    clean_df = clean_df[valid_cols]

    vif_rows = []
    values = clean_df.values.astype(float)

    for i, col in enumerate(clean_df.columns):
        vif_value = variance_inflation_factor(values, i)
        if vif_value < 5:
            interpretation = "Aceptable"
        elif vif_value < 10:
            interpretation = "Advertencia"
        else:
            interpretation = "Multicolinealidad alta"

        vif_rows.append({
            "variable": col,
            "vif": round(float(vif_value), 4),
            "interpretation": interpretation
        })

    return pd.DataFrame(vif_rows).sort_values("vif", ascending=False).reset_index(drop=True)

vif_candidate_vars = [
    col for col in ["units", "price", "revenue", "promo", "inventory", "avg_temp",
                    "topic1", "topic2", "topic3", "topic4", "topic5", "img_mean", "img_std"]
    if col in retail_df.columns
]

vif_results_df = compute_vif(retail_df[vif_candidate_vars])

display(Markdown("### Resultados de VIF"))
display(vif_results_df)


## Decisión profesional de selección de variables

Con base en criterios de negocio, correlación y VIF, se propone la siguiente decisión de selección:

### Variables que se conservan
- `units`
- `price`
- `promo`
- `inventory`
- `store_id`
- `product_id`
- `category`
- `topic1`
- `topic3`
- `topic4`

### Variables que se eliminan
- `revenue` por redundancia matemática
- `img_mean`, `img_std` por colinealidad extrema o baja utilidad analítica
- `avg_temp` por irrelevancia en este escenario
- `topic2`, `topic5` por mayor colinealidad relativa frente a otros componentes


In [ ]:
# ============================================================
# 4.1 Selección profesional de variables
# ============================================================

selected_numeric_vars = [col for col in ["units", "price", "promo", "inventory", "topic1", "topic3", "topic4"] if col in retail_df.columns]
selected_categorical_vars = [col for col in ["store_id", "product_id", "category"] if col in retail_df.columns]
selected_temporal_vars = [col for col in ["date", "day_of_week"] if col in retail_df.columns]

selected_variables_df = pd.DataFrame({
    "selected_variable": selected_temporal_vars + selected_categorical_vars + selected_numeric_vars
})

retail_selected_df = retail_df[selected_temporal_vars + selected_categorical_vars + selected_numeric_vars].copy()

display(Markdown("### Variables seleccionadas"))
display(selected_variables_df)

display(Markdown("### Dataset seleccionado"))
display(retail_selected_df.head())


## Paso 5. Codificación de variables categóricas

Los algoritmos de análisis y machine learning no pueden operar directamente con texto.  
Por ello, las variables categóricas deben convertirse en una representación numérica.

En este caso se utilizará **One-Hot Encoding** con `drop_first=True`, lo cual:
- evita el *dummy trap*,
- mantiene buena interpretabilidad,
- y facilita la construcción de pipelines escalables.


In [ ]:
# ============================================================
# 5. Codificación One-Hot
# ============================================================

retail_encoded_df = retail_selected_df.copy()

# Convertir fecha a variable numérica adicional opcional para preservar valor analítico
if "date" in retail_encoded_df.columns:
    retail_encoded_df["date_ordinal"] = retail_encoded_df["date"].map(pd.Timestamp.toordinal)
    retail_encoded_df = retail_encoded_df.drop(columns=["date"])

categorical_to_encode = [col for col in ["day_of_week", "store_id", "product_id", "category"] if col in retail_encoded_df.columns]

retail_encoded_df = pd.get_dummies(
    retail_encoded_df,
    columns=categorical_to_encode,
    drop_first=True,
    dtype=int
)

display(Markdown("### Dataset codificado"))
display(retail_encoded_df.head())

display(Markdown("### Dimensiones tras codificación"))
print(retail_encoded_df.shape)


## Paso 6. Normalización (Min-Max Scaling)

La normalización reescala los valores al intervalo `[0, 1]`.

Se recomienda en escenarios donde:
- el algoritmo depende de distancias,
- existen diferencias fuertes de escala,
- o se trabajará con redes neuronales y métodos sensibles al rango.

Aquí se aplica sobre las variables numéricas seleccionadas.


In [ ]:
# ============================================================
# 6. Normalización Min-Max
# ============================================================

retail_minmax_df = retail_encoded_df.copy()

numeric_to_scale = retail_minmax_df.select_dtypes(include=[np.number]).columns.tolist()

minmax_scaler = MinMaxScaler()
retail_minmax_df[numeric_to_scale] = minmax_scaler.fit_transform(retail_minmax_df[numeric_to_scale])

display(Markdown("### Dataset normalizado con Min-Max"))
display(retail_minmax_df.head())

display(Markdown("### Verificación de rangos (mínimo y máximo)"))
range_check_df = pd.DataFrame({
    "variable": numeric_to_scale,
    "min_value": retail_minmax_df[numeric_to_scale].min().values,
    "max_value": retail_minmax_df[numeric_to_scale].max().values
})
display(range_check_df.head(15))


## Paso 7. Estandarización (Z-score)

La estandarización transforma las variables para que tengan:
- media cercana a 0,
- desviación estándar cercana a 1.

Este enfoque es especialmente útil cuando se utilizarán:
- PCA,
- regresión lineal,
- SVM,
- K-means,
- y otros modelos paramétricos.

En el caso de RetailMax, esta es la opción más coherente cuando el siguiente paso es construir pipelines analíticos escalables y comparables.


In [ ]:
# ============================================================
# 7. Estandarización Z-score
# ============================================================

retail_standardized_df = retail_encoded_df.copy()

standard_scaler = StandardScaler()
retail_standardized_df[numeric_to_scale] = standard_scaler.fit_transform(retail_standardized_df[numeric_to_scale])

display(Markdown("### Dataset estandarizado"))
display(retail_standardized_df.head())

display(Markdown("### Validación de media y desviación estándar"))
standard_check_df = pd.DataFrame({
    "variable": numeric_to_scale,
    "mean": retail_standardized_df[numeric_to_scale].mean().round(4).values,
    "std": retail_standardized_df[numeric_to_scale].std().round(4).values
})
display(standard_check_df.head(15))


## ¿Normalización o estandarización?

### Uso recomendado
- **Normalización (0–1)**: KNN, clustering, redes neuronales
- **Estandarización (Z-score)**: PCA, SVM, regresión lineal, modelos paramétricos

No se recomienda aplicar ambas técnicas de forma redundante sobre el mismo grupo de variables en un pipeline final.  
No obstante, sí es válido construir versiones diferentes del dataset para comparar su desempeño con distintos algoritmos.


In [ ]:
# ============================================================
# 8. Resumen comparativo de técnicas
# ============================================================

scaling_recommendations_df = pd.DataFrame([
    {"technique": "Normalización (0-1)", "recommended_use": "KNN, clustering, redes neuronales"},
    {"technique": "Estandarización (Z-score)", "recommended_use": "PCA, SVM, regresión lineal, modelos paramétricos"},
    {"technique": "Ambas", "recommended_use": "Solo en pipelines diferenciados por tipo de variable o modelo"}
])

display(Markdown("### Recomendaciones de escalamiento"))
display(scaling_recommendations_df)


## Paso 8. Exportación de archivos

Para facilitar prácticas posteriores, se exportarán:
- la matriz de correlación,
- los resultados de VIF,
- la lista de variables seleccionadas,
- el dataset seleccionado,
- el dataset codificado,
- el dataset normalizado,
- y el dataset estandarizado.


In [ ]:
# ============================================================
# 9. Exportación de resultados
# ============================================================

correlation_matrix.to_csv("correlation_matrix.csv", index=True)
vif_results_df.to_csv("vif_results.csv", index=False)
selected_variables_df.to_csv("selected_variables.csv", index=False)
retail_selected_df.to_csv("retailmax_selected.csv", index=False)
retail_encoded_df.to_csv("retailmax_encoded.csv", index=False)
retail_minmax_df.to_csv("retailmax_minmax.csv", index=False)
retail_standardized_df.to_csv("retailmax_standardized.csv", index=False)

display(Markdown("### Archivos generados"))
print("- correlation_matrix.csv")
print("- vif_results.csv")
print("- selected_variables.csv")
print("- retailmax_selected.csv")
print("- retailmax_encoded.csv")
print("- retailmax_minmax.csv")
print("- retailmax_standardized.csv")

try:
    from google.colab import files
    files.download("correlation_matrix.csv")
    files.download("vif_results.csv")
    files.download("selected_variables.csv")
    files.download("retailmax_selected.csv")
    files.download("retailmax_encoded.csv")
    files.download("retailmax_minmax.csv")
    files.download("retailmax_standardized.csv")
except Exception:
    print("Si no estás en Google Colab, ignora este mensaje. Los archivos ya fueron creados en el directorio de trabajo.")


## Resumen final de la práctica

Este cuaderno:
1. carga el dataset integrado de RetailMax o genera una simulación de respaldo,
2. identifica variables numéricas y categóricas,
3. construye la matriz de correlación,
4. calcula VIF para evaluar multicolinealidad,
5. selecciona variables con base en criterios estadísticos y de negocio,
6. codifica variables categóricas mediante One-Hot Encoding,
7. genera una versión normalizada y otra estandarizada del dataset,
8. y exporta todos los archivos resultantes para prácticas posteriores.

Con ello, se obtiene una base preparada para modelado profesional y análisis posterior.
